# Reconnaître un schéma d'argumentation — la table de Walton et un classifieur déterministe

**Notebook pédagogique CoursIA** · #1961 Phase 5 (assets CoursIA réutilisables) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Corpus-free : tous les exemples sont synthétiques, domaine-public. Aucun LLM, aucune JVM — le classifieur est un matcher lexical déterministe.

Ce que couvre ce notebook :

1. les **10 schémas d'argumentation** (adaptation fidèle d'un livrable étudiant ; questions critiques canonales de Walton ajoutées) ;
2. la **classification déterministe** d'un texte vers son schéma (paires de mots-clés, précision d'abord) ;
3. l'**honnêteté du `None`** : quand aucun schéma ne se détache, le moteur le dit au lieu d'inventer ;
4. les **questions critiques** comme outil de réfutation ;
5. le pont vers le débat LLM (`schemes_as_prompt_context`).

**Garde round-trip** : chaque classification affichée ici est rejouée contre le moteur source par `tests/unit/coursia/argumentation_schemes/test_argumentation_schemes_roundtrip.py` (mêmes exemples, même JSON partagé) — ce notebook ne peut pas dériver silencieusement.


## §1 — Le modèle : 10 schémas, une force a priori, des questions critiques

Chaque schéma porte des **prémisses types**, une **conclusion type**, une **force a priori** (un prior sur la vigueur du schéma, pas un score mesuré) et des **questions critiques** — le test canonal de Walton qu'un contradicteur pose pour stresser le schéma.


In [1]:
# Imports + configuration. Le classifieur est déterministe (aucun LLM, aucune JVM).
import json
import logging
import os
import sys

logging.disable(logging.INFO)  # sorties propres malgré la chaîne d'import du dépôt

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
ROOT = _cwd if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")) else os.getcwd()
sys.path.insert(0, ROOT)

from argumentation_analysis.agents.core.debate.argumentation_schemes import (
    _load_argumentation_schemes,
    classify_scheme,
    schemes_as_prompt_context,
)

EXAMPLES_PATH = os.path.join(
    ROOT, "docs", "coursia_contrib", "argumentation_schemes_examples.json"
)

schemes = _load_argumentation_schemes()
print(f"{len(schemes)} schémas chargés.\n")
print(f"{'clé':<26}{'force':>6}   {'label':<46}{'questions':>10}")
print("-" * 90)
for s in schemes.values():
    print(f"{s.key:<26}{s.strength:>6.2f}   {s.label:<46}{len(s.critical_questions):>10}")


10 schémas chargés.

clé                        force   label                                          questions
------------------------------------------------------------------------------------------
modus_ponens                1.00   Déduction (modus ponens)                               2
expert_opinion              0.80   Argument d'autorité (advice of an expert)              3
analogy                     0.60   Argument par analogie                                  2
cause_effect                0.70   Argument de cause à effet                              2
consensus                   0.85   Argument d'ad hominem consensuel (appeal to consensus)         2
empirical_evidence          0.90   Argument empirique (from evidence)                     2
economic_argument           0.75   Argument économique (cost-benefit)                     2
precautionary_principle     0.70   Principe de précaution                                 2
moral_argument              0.80   Argument moral (f

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-09-14 17:20:42 [WARNING] [Services.CryptoService] crypto_service.__init__:46 - Service de chiffrement initialisé sans clé. Le chiffrement est désactivé.


## §2 — Classifications réelles : le matcher départage dix textes

Dix exemples synthétiques, un par schéma. Chacun passe dans le moteur réel — et est **vérifié par assertion** : si le schéma attendu ne sort pas, la cellule échoue. Ce notebook est un instrument, pas une vitrine.


In [2]:
# Les dix exemples (JSON partagé avec la garde round-trip) passent au classifieur réel.
with open(EXAMPLES_PATH, encoding="utf-8") as fh:
    examples = json.load(fh)

for ex in examples["positives"]:
    scheme = classify_scheme(ex["text"])
    assert scheme is not None and scheme.key == ex["expected_key"], (ex, scheme)
    print(f"« {ex['text']} »")
    print(f"   -> {scheme.label}  ({scheme.key}, force a priori {scheme.strength:.2f})")
    print(f"   signature : {ex['why']}\n")

print("10/10 — chaque texte rend le schéma attendu.")


« Selon un spécialiste reconnu, la molécule est sans danger pour l'usage concerné. »
   -> Argument d'autorité (advice of an expert)  (expert_opinion, force a priori 0.80)
   signature : La paire « selon » + « spécialiste » signe l'argument d'autorité.

« L'étude repose sur une mesure directe et un échantillon représentatif de la population. »
   -> Argument empirique (from evidence)  (empirical_evidence, force a priori 0.90)
   signature : La paire « étude » + « mesure » signe l'argument empirique.

« La majorité des chercheurs parviennent à un accord sur ce point. »
   -> Argument d'ad hominem consensuel (appeal to consensus)  (consensus, force a priori 0.85)
   signature : La paire « majorité » + « accord » signe l'appel au consensus.

« À l'instar du pilote maritime, comparable à un navigateur solitaire, le gestionnaire décide seul. »
   -> Argument par analogie  (analogy, force a priori 0.60)
   signature : La paire « à l'instar » + « comparable » signe l'analogie.

« Cette cause 

## §3 — L'honnêteté du `None` : quatre textes sans schéma

Le matcher exige la **paire complète** de mots-clés — « expert » seul ne suffit pas, il manque « domaine » ou « source ». Précision plutôt que rappel : le moteur préfère `None` à une étiquette fabriquée. Un texte vide retourne `None` immédiatement.


In [3]:
for ex in examples["negatives"]:
    scheme = classify_scheme(ex["text"])
    assert scheme is None, (ex, scheme)
    label = ex["text"] if ex["text"] else "(texte vide)"
    print(f"None <- « {label} »")
    print(f"   pourquoi : {ex['why']}\n")

print("4/4 — aucun schéma fabriqué.")


None <- « Bonjour, il fait beau aujourd'hui. »
   pourquoi : Aucune signature lexicale d'aucun schéma.

None <- « Le rapport compte douze pages et trois annexes. »
   pourquoi : Description neutre, non argumentative.

None <- « L'expert est attendu à la conférence. »
   pourquoi : Mot-clé isolé (« expert » sans « domaine » ni « source ») : le classifieur exige la paire complète — précision plutôt que rappel.

None <- « (texte vide) »
   pourquoi : Texte vide : retour immédiat None, jamais d'étiquette fabriquée.

4/4 — aucun schéma fabriqué.


## §4 — Les questions critiques : l'outil de réfutation du schéma

Classer **nomme** le schéma ; les questions critiques le **stressent**. Prenons l'argument empirique : ses questions canonales attaquent la fiabilité des données et la représentativité de l'échantillon — exactement là où un argument empirique se défend mal.


In [4]:
# L'exemple empirique : schéma, prémisses types, et le test canonal de Walton.
ex = examples["positives"][1]
scheme = classify_scheme(ex["text"])
print(f"Texte            : {ex['text']}")
print(f"Schéma           : {scheme.label} (force {scheme.strength:.2f})")
print(f"Prémisses types  : {scheme.premises_pattern}")
print(f"Conclusion type  : {scheme.conclusion_pattern}")
print("Questions critiques :")
for i, q in enumerate(scheme.critical_questions, 1):
    print(f"  {i}. {q}")


Texte            : L'étude repose sur une mesure directe et un échantillon représentatif de la population.
Schéma           : Argument empirique (from evidence) (force 0.90)
Prémisses types  : ['Data shows P', 'Data is reliable', 'Sample is representative']
Conclusion type  : P is supported by evidence
Questions critiques :
  1. Les données sont-elles fiables (collecte, mesure) ?
  2. L'échantillon est-il représentatif de la population visée ?


## §5 — Le pont vers le débat LLM : la table comme contexte de prompt

Côté LLM, la même table est rendue en bloc de connaissances (`schemes_as_prompt_context`) : un échange de débat peut alors citer le schéma **et** les questions qui le testent, au lieu d'une étiquette flottante.


In [5]:
print(schemes_as_prompt_context())


  1. « Déduction (modus ponens) » (force a priori 1.00) — questions critiques de test : La prémisse P est-elle effectivement établie ? ; L'implication P → Q est-elle valide (pas un sophisme conditionnel) ?
  2. « Argument d'autorité (advice of an expert) » (force a priori 0.80) — questions critiques de test : E est-elle réellement une source experte sur ce domaine ? ; L'avis de E est-il cohérent avec le consensus des autres experts ?
  3. « Argument par analogie » (force a priori 0.60) — questions critiques de test : En quoi les cas A et B sont-ils réellement similaires sur la dimension pertinente ? ; Existe-t-il une différence pertinente qui brise l'analogie ?
  4. « Argument de cause à effet » (force a priori 0.70) — questions critiques de test : La relation causale A → B est-elle établie (et non une simple corrélation) ? ; Y a-t-il d'autres causes possibles de B ?
  5. « Argument d'ad hominem consensuel (appeal to consensus) » (force a priori 0.85) — questions critiques de test : Le

## Ce qu'il faut retenir

- **10 schémas de Walton** : prémisses types, conclusion type, force a priori, questions critiques canonales ;
- un **classifieur lexical déterministe** — paires de mots-clés (accentuées), schémas spécifiques d'abord, `modus_ponens` en dernier car son signal (« si… par conséquent… ») est le plus faible ;
- **`None` est un résultat honnête** — précision plutôt que rappel, jamais d'étiquette fabriquée ;
- la même table **alimente le prompt LLM** du débat via `schemes_as_prompt_context`.

Ce notebook est exécuté — les sorties ci-dessus sont réelles, produites par le moteur du dépôt, pas retouchées.

**Références** : Walton, *Argumentation Schemes* (questions critiques canonales) · moteur : `argumentation_analysis/agents/core/debate/argumentation_schemes.py` (restauration fidèle du livrable étudiant `1_2_7_argumentation_dialogique`) · issue #1961 Phase 5 · garde : `tests/unit/coursia/argumentation_schemes/test_argumentation_schemes_roundtrip.py`.
